# AEI Indonesia Diagnostic

This notebook downloads the Anthropic Economic Index September 2025 geographic release and answers four questions needed to decide what Indonesia analysis is feasible:

1. **How many Indonesia rows exist** — sample size determines what's analyzable
2. **Which facets does Indonesia appear in** — is task-level data available, or only aggregates?
3. **Which tasks/clusters cleared the privacy threshold for Indonesia** — your actual workable universe
4. **What's Indonesia's AUI value in the data** — sanity check against published 0.36

Run this end-to-end before committing to any Indonesia-focused thesis direction.

**Requires:** `pip install pandas huggingface_hub`

## 0. Setup

In [1]:
# Uncomment if needed
# !pip install pandas huggingface_hub

import sys
from pathlib import Path
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

REPO_ID = 'Anthropic/EconomicIndex'
RELEASE = 'release_2025_09_15'
LOCAL_DIR = Path('./aei_data')
LOCAL_DIR.mkdir(exist_ok=True)

# Indonesia identifiers — data may use ISO-2 (raw) or ISO-3 (enriched)
INDONESIA_CODES = ['ID', 'IDN', 'Indonesia']

print('Setup complete.')

Setup complete.


## 1. List files in the September 2025 release

In [2]:
try:
    all_files = list_repo_files(REPO_ID, repo_type='dataset')
    release_files = sorted([f for f in all_files if RELEASE in f])
    print(f'Found {len(release_files)} files in {RELEASE}/:\n')
    for f in release_files:
        print(f'  {f}')
except Exception as e:
    print(f'Could not list files: {e}')
    print('Will try direct downloads with known filenames.')
    release_files = []

Found 38 files in release_2025_09_15/:

  release_2025_09_15/README.md
  release_2025_09_15/code/aei_analysis_functions_1p_api.py
  release_2025_09_15/code/aei_analysis_functions_claude_ai.py
  release_2025_09_15/code/aei_report_v3_analysis_1p_api.ipynb
  release_2025_09_15/code/aei_report_v3_analysis_claude_ai.ipynb
  release_2025_09_15/code/aei_report_v3_change_over_time_claude_ai.py
  release_2025_09_15/code/aei_report_v3_preprocessing_claude_ai.ipynb
  release_2025_09_15/code/preprocess_gdp.py
  release_2025_09_15/code/preprocess_iso_codes.py
  release_2025_09_15/code/preprocess_onet.py
  release_2025_09_15/code/preprocess_population.py
  release_2025_09_15/data/input/BTOS_National.xlsx
  release_2025_09_15/data/input/Population by single age _20250903072924.csv
  release_2025_09_15/data/input/automation_vs_augmentation_v1.csv
  release_2025_09_15/data/input/automation_vs_augmentation_v2.csv
  release_2025_09_15/data/input/bea_us_state_gdp_2024.csv
  release_2025_09_15/data/input/c

## 2. Download the Claude.ai geographic CSV

In [3]:
# Find the raw Claude.ai CSV (filename may vary slightly between releases)
candidates = [f for f in release_files if 'claude_ai' in f and f.endswith('.csv')]
if not candidates:
    # Fallback to known filename from documentation
    candidates = [f'{RELEASE}/aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv']

geo_file_path = None
for candidate in candidates:
    try:
        print(f'Downloading: {candidate}')
        path = hf_hub_download(
            repo_id=REPO_ID,
            filename=candidate,
            repo_type='dataset',
            local_dir=LOCAL_DIR,
        )
        geo_file_path = path
        print(f'  -> Saved to: {path}')
        break
    except Exception as e:
        print(f'  Failed: {e}')

if not geo_file_path:
    raise RuntimeError('Could not download geographic data. Check network or repo access.')

Downloading: release_2025_09_15/data/intermediate/aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv
  -> Saved to: aei_data\release_2025_09_15\data\intermediate\aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv


In [4]:
# Try to grab the documentation file too
try:
    doc_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=f'{RELEASE}/data_documentation.md',
        repo_type='dataset',
        local_dir=LOCAL_DIR,
    )
    print(f'Documentation saved to: {doc_path}')
except Exception:
    print('(data_documentation.md not retrieved — non-critical)')

Documentation saved to: aei_data\release_2025_09_15\data_documentation.md


## 3. Inspect the schema

Before doing anything else, look at every column, its type, and a sample value. This is your ground truth.

In [5]:
geo = pd.read_csv(geo_file_path)
print(f'Total rows: {len(geo):,}')
print(f'Total columns: {len(geo.columns)}')
print(f'\nColumns and dtypes:')
for col in geo.columns:
    n_unique = geo[col].nunique()
    sample = geo[col].dropna().iloc[0] if geo[col].notna().any() else 'N/A'
    print(f'  {col:30s}  dtype={str(geo[col].dtype):10s}  unique={n_unique:>7,}  sample={str(sample)[:50]}')

Total rows: 100,062
Total columns: 10

Columns and dtypes:
  geo_id                          dtype=object      unique=    202  sample=AD
  geography                       dtype=object      unique=      3  sample=country
  date_start                      dtype=object      unique=      1  sample=2025-08-04
  date_end                        dtype=object      unique=      1  sample=2025-08-11
  platform_and_product            dtype=object      unique=      1  sample=Claude AI (Free and Pro)
  facet                           dtype=object      unique=      7  sample=collaboration
  level                           dtype=int64       unique=      3  sample=0
  variable                        dtype=object      unique=     12  sample=collaboration_count
  cluster_name                    dtype=object      unique= 14,713  sample=not_classified
  value                           dtype=float64     unique= 23,798  sample=25.0


In [6]:
# Peek at the first few rows
geo.head(10)

,geo_id,geography,date_start,date_end,platform_and_product,facet,level,variable,cluster_name,value
0,AD,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),collaboration,0,collaboration_count,not_classified,25.000000
1,AD,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),collaboration,0,collaboration_pct,not_classified,62.500000
2,AD,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),collaboration,0,collaboration_count,task iteration,15.000000
3,AD,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),collaboration,0,collaboration_pct,task iteration,37.500000
4,AD,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),country,0,usage_count,NaN,40.000000
5,AD,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),country,0,usage_pct,NaN,0.004147
6,AE,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),collaboration,0,collaboration_count,directive,1200.000000
7,AE,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),collaboration,0,collaboration_pct,directive,39.853869
8,AE,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),collaboration,0,collaboration_count,feedback loop,349.000000
9,AE,country,2025-08-04,2025-08-11,Claude AI (Free and Pro),collaboration,0,collaboration_pct,feedback loop,11.590834


## 4. Diagnostic 1 — Indonesia row count

In [7]:
# Find which column holds country codes
geo_col_candidates = ['geo_id', 'country', 'country_code', 'geography', 'iso_code']
geo_col = None
for c in geo_col_candidates:
    if c in geo.columns:
        geo_col = c
        break

if geo_col is None:
    # Inspect string columns for ISO codes
    print('No standard geo column found. Searching for ISO codes...')
    for col in geo.select_dtypes(include='object').columns:
        sample_vals = geo[col].dropna().unique()[:5]
        if any(v in ['US', 'USA', 'ID', 'IDN', 'GB', 'DE'] for v in sample_vals):
            geo_col = col
            print(f'  Found country codes in column: {col}')
            break

if geo_col is None:
    raise RuntimeError('Could not identify country column. Inspect schema above and set geo_col manually.')

print(f"Using column: '{geo_col}'")

id_mask = geo[geo_col].isin(INDONESIA_CODES)
id_rows = geo[id_mask]

print(f'\nIndonesia rows: {len(id_rows):,}')
print(f'Indonesia codes found: {id_rows[geo_col].unique().tolist()}')
print(f'Total unique countries in data: {geo[geo_col].nunique()}')

Using column: 'geo_id'

Indonesia rows: 1,506
Indonesia codes found: ['ID']
Total unique countries in data: 202


In [8]:
# Compare Indonesia's row count to a high-AUI country and a peer
benchmarks = {
    'US/USA': ['US', 'USA'],
    'India': ['IN', 'IND'],
    'Israel': ['IL', 'ISR'],
    'Indonesia': INDONESIA_CODES,
}

print('Row count comparison:')
for label, codes in benchmarks.items():
    n = (geo[geo_col].isin(codes)).sum()
    print(f'  {label:12s}: {n:,} rows')

Row count comparison:
  US/USA      : 3,964 rows
  India       : 2,564 rows
  Israel      : 1,910 rows
  Indonesia   : 1,506 rows


## 5. Diagnostic 2 — Which facets does Indonesia appear in?

In [9]:
facet_col = None
for c in ['facet', 'facet_name', 'dimension', 'metric_type']:
    if c in geo.columns:
        facet_col = c
        break

if facet_col:
    print(f"Using facet column: '{facet_col}'\n")
    id_facets = id_rows[facet_col].value_counts()
    print(f'Indonesia row counts by facet:')
    for facet, n in id_facets.items():
        print(f'  {str(facet):40s} {n:>6,} rows')
else:
    print("No 'facet' column found. Inspect schema above to identify the dimension column.")

Using facet column: 'facet'

Indonesia row counts by facet:
  request                                   1,138 rows
  onet_task                                   340 rows
  collaboration                                24 rows
  country                                       2 rows
  state_us                                      2 rows


In [10]:
# Compare global facet coverage to Indonesia's coverage
if facet_col:
    print('All facets in dataset (Indonesia coverage marked):')
    all_facets = geo[facet_col].value_counts()
    id_facets = id_rows[facet_col].value_counts()
    for facet, n in all_facets.head(20).items():
        id_n = id_facets.get(facet, 0)
        marker = '[Y]' if id_n > 0 else '[ ]'
        print(f'  {marker} {str(facet):40s} {n:>8,} global  | {id_n:>5,} Indonesia')

All facets in dataset (Indonesia coverage marked):
  [Y] request                                    50,346 global  | 1,138 Indonesia
  [Y] onet_task                                  24,080 global  |   340 Indonesia
  [ ] onet_task::collaboration                   14,454 global  |     0 Indonesia
  [ ] request::collaboration                      8,508 global  |     0 Indonesia
  [Y] collaboration                               2,224 global  |    24 Indonesia
  [Y] country                                       346 global  |     2 Indonesia
  [Y] state_us                                      104 global  |     2 Indonesia


## 6. Diagnostic 3 — Which tasks/clusters cleared the threshold for Indonesia?

In [11]:
cluster_col = None
for c in ['cluster_name', 'cluster', 'task', 'category', 'value']:
    if c in geo.columns:
        cluster_col = c
        break

print(f'Cluster column: {cluster_col}')

if cluster_col and facet_col:
    print(f'\nUnique cluster values per facet (Indonesia only):\n')
    for facet in id_rows[facet_col].unique():
        subset = id_rows[id_rows[facet_col] == facet]
        n_clusters = subset[cluster_col].nunique()
        print(f'  {str(facet):40s} {n_clusters:>4} unique values')

Cluster column: cluster_name

Unique cluster values per facet (Indonesia only):

  collaboration                               7 unique values
  country                                     0 unique values
  onet_task                                 166 unique values
  request                                   527 unique values
  state_us                                    0 unique values


In [12]:
# Show the actual top tasks/requests for Indonesia
if cluster_col and facet_col:
    task_facets = [f for f in id_rows[facet_col].unique()
                   if any(k in str(f).lower() for k in ['task', 'onet', 'request'])]
    
    if task_facets:
        primary_task_facet = task_facets[0]
        task_rows = id_rows[id_rows[facet_col] == primary_task_facet].copy()
        
        print(f"Top 20 Indonesian tasks/requests in facet '{primary_task_facet}':\n")
        sort_col = 'usage_pct' if 'usage_pct' in task_rows.columns else 'count'
        if sort_col in task_rows.columns:
            top = task_rows.nlargest(20, sort_col)[[cluster_col, sort_col]]
            for _, r in top.iterrows():
                val = r[sort_col]
                val_str = f'{val:.3f}' if val < 1 else f'{val:,.0f}'
                print(f'  {val_str:>10s}  {str(r[cluster_col])[:80]}')
        else:
            print('(no usage_pct or count column to sort by)')
            display(task_rows[[cluster_col]].head(20))
    else:
        print('No task-level facet identified for Indonesia.')
        print('Indonesia data is limited to aggregate dimensions.')

Top 20 Indonesian tasks/requests in facet 'onet_task':

(no usage_pct or count column to sort by)


,cluster_name
19720,adapt instructional content or delivery method...
19721,adapt instructional content or delivery method...
19722,"address the relationships of quantities, magni..."
19723,"address the relationships of quantities, magni..."
19724,advise students on academic and vocational cur...
19725,advise students on academic and vocational cur...
19726,advise students on academic and vocational cur...
19727,advise students on academic and vocational cur...
19728,analyze characteristics of animals to identify...
19729,analyze characteristics of animals to identify...


## 7. Diagnostic 4 — Indonesia's AUI value

Should be approximately **0.36** based on the published September 2025 report. If your data shows something different, that itself is a finding worth investigating.

In [13]:
aui_col = None
for c in ['usage_per_capita_index', 'aui', 'AUI', 'per_capita_index']:
    if c in geo.columns:
        aui_col = c
        break

if aui_col:
    id_aui_vals = id_rows[aui_col].dropna().unique()
    print(f'AUI values found for Indonesia: {id_aui_vals}')
    print('(should be a single value around 0.36 per published reports)')
    
    # Compute global distribution for context
    all_aui = geo.dropna(subset=[aui_col]).groupby(geo_col)[aui_col].first()
    print(f'\nGlobal AUI distribution:')
    print(f'  Median: {all_aui.median():.3f}')
    print(f'  Mean:   {all_aui.mean():.3f}')
    print(f'  Min:    {all_aui.min():.3f} ({all_aui.idxmin()})')
    print(f'  Max:    {all_aui.max():.3f} ({all_aui.idxmax()})')
    print(f'  Countries with AUI: {len(all_aui)}')
else:
    print('No AUI column found. Look for usage_per_capita_index or similar.')

No AUI column found. Look for usage_per_capita_index or similar.


In [14]:
# Plot Indonesia's position in the global AUI distribution
if aui_col:
    import matplotlib.pyplot as plt
    
    aui_sorted = all_aui.sort_values()
    median_aui = aui_sorted.median()
    
    id_country = next((c for c in INDONESIA_CODES if c in aui_sorted.index), None)
    
    fig, ax = plt.subplots(figsize=(13, 5))
    colors = ['#D85A30' if c == id_country else '#B4B2A9' for c in aui_sorted.index]
    ax.bar(range(len(aui_sorted)), aui_sorted.values, color=colors, edgecolor='white', linewidth=0.3)
    ax.axhline(median_aui, color='#534AB7', linestyle='--', linewidth=1.5,
               label=f'Global median: {median_aui:.2f}')
    ax.axhline(1.0, color='#666', linestyle=':', linewidth=1, label='AUI = 1.0 (proportional)')
    ax.set_yscale('log')
    ax.set_xlabel('Countries (sorted by AUI)')
    ax.set_ylabel('AUI (log scale)')
    ax.set_title('Global AUI distribution — Indonesia highlighted in orange')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 8. Feasibility verdict

In [15]:
n_id = len(id_rows)
n_facets = id_rows[facet_col].nunique() if facet_col else 0
n_task_clusters = 0
if cluster_col and facet_col:
    task_facets = [f for f in id_rows[facet_col].unique()
                   if any(k in str(f).lower() for k in ['task', 'onet'])]
    if task_facets:
        n_task_clusters = id_rows[id_rows[facet_col].isin(task_facets)][cluster_col].nunique()

print(f'Indonesia rows total:        {n_id:,}')
print(f'Indonesia facets present:    {n_facets}')
print(f'Indonesia task-level values: {n_task_clusters}')

print('\nFeasibility verdict:')
if n_id < 50:
    print('  Sample is very small. Stick to AUI + aggregate automation share only.')
    print('  Indonesia analysis viable as a single descriptive chapter, not a full thesis.')
elif n_task_clusters < 20:
    print('  Limited task resolution. SOC major group level analysis is doable.')
    print('  Per-task or per-occupation analysis is not reliable.')
elif n_task_clusters < 100:
    print('  Moderate task resolution. SOC major group + selected occupations work.')
    print('  Be cautious with rare-task claims.')
else:
    print('  Good task resolution. Full occupation-level analysis is feasible.')
    print('  This supports a thesis-scale Indonesia analysis.')

print('\nDone. Review the outputs above before deciding on the next step.')

Indonesia rows total:        1,506
Indonesia facets present:    5
Indonesia task-level values: 166

Feasibility verdict:
  Good task resolution. Full occupation-level analysis is feasible.
  This supports a thesis-scale Indonesia analysis.

Done. Review the outputs above before deciding on the next step.
